<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 2

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
import warnings
import numpy as np

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df2 = pd.read_csv('Dataset 2.csv')

# Drops all unnecessary columns
df2 = df2.drop(columns=['age']) 

# Drops all with null values
df2.dropna(inplace=True)

# Remove Outliers
upper_quantile = df2['price'].quantile(0.99)
df2 = df2[df2['price'] <= upper_quantile]

# Log transforms price to reduce variablilty
df2['price'] = np.log(df2['price'])

# One-Hot encode all categorical features
categorical_cols_to_encode = df2.select_dtypes(include='object').columns
df2 = pd.get_dummies(df2, columns=categorical_cols_to_encode, drop_first=True)

# Find the target and numerical features
target_column = 'price'
numerical_features = ['area', 'bhk', 'bathroom']
y = df2[target_column].copy()

# Create scalar
scaler = StandardScaler()

# Create feature base without price
x_base = df2.drop(columns=[target_column]).copy()

x_linear = x_base.copy()

# Apply polynomial features to numerical columns for linear model
poly = PolynomialFeatures(degree=2, include_bias=False)
numerical_poly_features = poly.fit_transform(x_linear[numerical_features])
poly_feature_names = poly.get_feature_names_out(numerical_features)
x_linear_poly_df = pd.DataFrame(numerical_poly_features, columns=poly_feature_names, index=x_linear.index)

# Drop original numerical columns and concatenate polynomial features for linear model
x_linear = x_linear.drop(columns=numerical_features)
x_linear = pd.concat([x_linear, x_linear_poly_df], axis=1)

# Find all numerical features, even the created ones for linear model
all_numerical_features_linear = x_linear.select_dtypes(include=np.number).columns.tolist()

# Scale numerical features for linear model
x_linear[all_numerical_features_linear] = scaler.fit_transform(x_linear[all_numerical_features_linear])

x_tree = x_base.copy()

# Scale numerical features for tree models
x_tree[numerical_features] = scaler.fit_transform(x_tree[numerical_features])

# Splits the data 80/20 for training and testing the model respectfully
x_train_linear, x_test_linear, y_train_linear, y_test_linear = train_test_split(x_linear, y, test_size=0.2, random_state=42)
x_train_tree, x_test_tree, y_train_tree, y_test_tree = train_test_split(x_tree, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 2: Dataset 2.csv")
print("=" * 60)
print("\n--- Features for Linear Models (x_linear) ---")
print("\nFirst 5 rows:")
display(x_linear.head(5))
print("\nData types:")
print(x_linear.dtypes)
print("\nMissing values:")
print(x_linear.isnull().sum().sum())
print(f"Shape of x_linear: {x_linear.shape}")

print("\n--- Features for Tree Models (x_tree) ---")
print("\nFirst 5 rows:")
display(x_tree.head(5))
print("\nData types:")
print(x_tree.dtypes)
print("\nMissing values:")
print(x_tree.isnull().sum().sum())
print(f"Shape of x_tree: {x_tree.shape}")

print("\nTarget variable (price) statistics:")
print(y.describe())

# Price is in Lahk which is 100,000 Rupees (₹)

In [ ]:
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def rmse_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate RMSE on the original scale
    return np.sqrt(mean_squared_error(y_true_original, y_pred_original))

def mae_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate MAE on the original scale
    return mean_absolute_error(y_true_original, y_pred_original)

def r2_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate R2 on the original scale
    return r2_score(y_true_original, y_pred_original)

# Create scorers that can be used with GridSearchCV or cross_val_score
# 'neg_' prefix is used for metrics to be minimized (RMSE, MAE)
neg_rmse_original_scorer = make_scorer(rmse_original_scale, greater_is_better=False)
neg_mae_original_scorer = make_scorer(mae_original_scale, greater_is_better=False)
r2_original_scorer = make_scorer(r2_original_scale, greater_is_better=True)

### Linear Regression for Dataset 2

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("\n -------Linear Regression Model-----")


#Initializing Linear Regression Model and Running
lr = LinearRegression()
lr.fit(x_train_linear, y_train_linear)
y_pred = lr.predict(x_test_linear)


#Fetching the Performance Metrics For The Model
rmse = rmse_original_scale(y_test_linear, y_pred)
mae = mae_original_scale(y_test_linear, y_pred)
r2 = r2_original_scale(y_test_linear, y_pred)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score

#Graph Showing predicted vs Actual With Errors and Coefficents

#Printing Performance Metrics for the model
print(f"Root Mean Squared Error: {rmse:.2f}\n")
print(f"Mean Absolute Error: {mae:.2f}\n")
print(f"R^2 Score: {r2:.2f}\n")
print(f"\nCross-validation RMSE:\t${-cross_val_score(lr, x_linear, y, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(lr, x_linear, y, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(lr, x_linear, y, cv=5, scoring=r2_original_scorer).mean():.4f}")

y_test_actual = np.exp(y_test_linear)
y_pred_actual = np.exp(y_pred)

#Constructing A Scatter Plot of Real Vs Expected Value to Vizualize the Perfomrance of the Model on The Test Set
plt.figure(figsize=(10, 6))
plt.scatter(y_test_actual, y_pred_actual, alpha=0.6, color='blue')
min_val = min(y_test_actual.min(), y_pred_actual.min())
max_val = max(y_test_actual.max(), y_pred_actual.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
plt.xlabel('Actual Prices (Lakh ₹)', fontsize=12)
plt.ylabel('Predicted Prices (Lakh ₹)', fontsize=12)
plt.title('Actual vs. Predicted Property Prices', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

In [ ]:
#Fetching Feature Names and Coefficents
feature_names = x_train_linear.columns
coefficients = lr.coef_

#Wrapping them together in a Pandas DataFrame for Easy Comparision and Viewing
feature_importance = pd.DataFrame({
 'Feature': feature_names,
 'Coefficient': coefficients
})

#Using the Absolute Function in the NumPy Library before Sorting the Coefficents in Decending Order
feature_importance['Abs_Coefficient'] = np.abs(feature_importance['Coefficient'])
top_10_features = feature_importance.sort_values(by='Abs_Coefficient', ascending=False).head(10)

#Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
 data=top_10_features, 
 x='Coefficient', 
 y='Feature', 
)
plt.title('Top 10 Features Driving Property Prices', fontsize=15)
plt.xlabel('Coefficient Value (Impact on Price in Lakh ₹)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features[['Feature', 'Coefficient']])


The Linear Regression model is able to predict the prices of cheap and standard homes very well, which accounts for its high R^2 score of 0.85. However, as the prices of the properties increase, the model begins to break down. The most important features seem to be the real estate companies themselves and their locations, as these were the features that made it into the top 10. This may indicate that for this specific dataset, "builder_Shree sakthivel realestate" and "builder_Vinay Asrani" may have made up a majority of the middle-to-high-end real estate, or perhaps they only had middle-to-high-end real estate within this sample.

The model then became dependent on these two features to make predictions on higher-cost housing, which may explain the low accuracy once the price of the houses crossed 200 Lakh. This suggests that a dataset with more high-end housing from a variety of real estate companies may be required. When the datasets are joined, we should expect the R^2 value to reduce, but the accuracy at the top of the price bracket should improve as the model finds more reliable predictors.This hypothesis is further backed up by a Mean Squared Error (MSE) that greatly exceeds the Mean Absolute Error (MAE). The MAE is fairly small and manageable since it doesn't penalize large deviations to the same extent as the MSE, which is orders of magnitude larger. Even when using the Root Mean Squared Error (RMSE), the value is still over double the MAE, further demonstrating the ineffectiveness of the model at the higher end of the dataset.

Therefore the core problems with the linear regression model for this dataset is the massive error at the top. In statistics you would remedy this with a log transformation. This "squishes" the data closer together and may allow the model to become less reliant on the real estate corporations themselves to predict the value of the house. Additionally a non linear model like RF or even a NN may help at the cost of increased computing cost. Additionally you could even go as far to sort the houses into groups(price ranges) and have the model focus on sorting the houses into these price ranges which should reduce the noise. 